# Package `__init__` files
Writes every `__init__.py` in the package except `devices/__init__.py`, which holds the driver
registry and has its own notebook (`devices_devNB`).

**Host only.** Nothing here talks to the board: every other notebook's hardware section
imports these files on MicroPython, which is their check there.

**Where each cell runs:** all of it in ordinary Python (CPython) on this computer.

# TODO:
Nothing open.

Decided (2026-11-16): `__init__` files are written with `%%writefile` (relative imports cannot
run as a bare cell). The package `__init__` imports only the core layer, so importing one
driver on the board does not pull every driver into RAM.

# Python file
location: `src/micropython_i2c/__init__.py`, `src/micropython_i2c/devices/*/__init__.py`,
`src/micropython_i2c/sim/__init__.py`

tests: `tests/test_package_inits.py`

# Imports
## External
None.

## Internal
None: the files below are written, not imported, until the checks at the end.

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_package_inits.py
import importlib

import pytest

Notebook-only: the imports the test cells use.

In [ ]:
import importlib

import pytest

# Intent
**Purpose:** the package's shape. Which names `import micropython_i2c` gives, and a docstring
for each subpackage so `help()` says what lives there.

```
micropython_i2c/            core layer: connection, nodes, protocols, helpers
  devices/                  the driver registry (devices_devNB)
    sensors/{environmental, magnetic, distance}/
    drivers/                outputs: relays, I/O expanders
    muxes/
  sim/                      host-only simulation: clock, signals, fake bus, bench
```

## What the files define

| File | What it is |
|---|---|
| `micropython_i2c/__init__.py` | the core names: `I2CConnection`, `I2CDevice`, `I2CMux`, protocols, helpers |
| `devices/<type>/__init__.py` | a docstring naming the parts in that folder |
| `sim/__init__.py` | a docstring for the simulation package |

# Code
## TODO:
Nothing

## `micropython_i2c/__init__.py`
The core layer only. Drivers are imported on demand (`devices.driver_class`), and `sim` is
never imported by board code.

In [ ]:
%%writefile ../src/micropython_i2c/__init__.py
"""PyVISA-style I2C for MicroPython: connections, nodes, protocols, drivers, routing.

Layers, bottom up::

    machine.I2C                        bus level, never hidden
    I2CConnection                      bound (bus, path, address); lock + routing
    I2CNode -> I2CDevice | I2CMux      datasheet facts + 4-method transport
    BitTools / CRCMixin                stateless helpers | checksum contract
    RegisterI2CBase / CommandI2CBase   protocol mixins, transport-agnostic
    devices.{sensors,drivers,muxes}.*  concrete parts, by type
    RoutingTable / I2CManager          config -> validated topology -> live objects
    sim.*                              host-only: simulated clock, signals, fake bus, bench

Devices live in ``micropython_i2c.devices`` (``sensors``, ``drivers``, ``muxes``) and are
imported on demand. See ``docs/design.md`` in the repo for the reasoning behind each layer.

Generated by ``dev_notebooks/package_inits_devNB``: edit the notebook, not this file.
"""

from .bits import BitTools
from .compat import NullLock, default_lock
from .connection import I2CConnection
from .crc import CRCMixin
from .node import I2CDevice, I2CMux, I2CNode
from .protocol import CommandI2CBase, RegisterI2CBase

__version__ = "0.1.0"

__all__ = [
    "BitTools",
    "CRCMixin",
    "CommandI2CBase",
    "I2CConnection",
    "I2CDevice",
    "I2CMux",
    "I2CNode",
    "NullLock",
    "RegisterI2CBase",
    "default_lock",
]

## Device subpackages
Docstring-only `__init__` files: each module in these folders is imported directly
(`from micropython_i2c.devices.sensors.environmental.shtc3 import SHTC3`). A part's fake
(`<part>_fake.py`) sits next to its driver and is host-only.

In [ ]:
%%writefile ../src/micropython_i2c/devices/sensors/__init__.py
"""Sensors, by kind: ``environmental``, ``magnetic``, ``distance``. Import each module directly.

Generated by ``dev_notebooks/package_inits_devNB``: edit the notebook, not this file.
"""

In [ ]:
%%writefile ../src/micropython_i2c/devices/sensors/environmental/__init__.py
"""Environmental sensors: TMP102 (temperature), SHTC3 (humidity, temperature).

Generated by ``dev_notebooks/package_inits_devNB``: edit the notebook, not this file.
"""

In [ ]:
%%writefile ../src/micropython_i2c/devices/sensors/magnetic/__init__.py
"""Magnetic sensors: TMAG5273 (3-axis Hall).

Generated by ``dev_notebooks/package_inits_devNB``: edit the notebook, not this file.
"""

In [ ]:
%%writefile ../src/micropython_i2c/devices/sensors/distance/__init__.py
"""Optical distance sensors (time of flight): VL53L1X, VL53L4CD.

Generated by ``dev_notebooks/package_inits_devNB``: edit the notebook, not this file.
"""

In [ ]:
%%writefile ../src/micropython_i2c/devices/drivers/__init__.py
"""Output drivers (relays, I/O expanders): QwiicRelay. Import each module directly.

Generated by ``dev_notebooks/package_inits_devNB``: edit the notebook, not this file.
"""

In [ ]:
%%writefile ../src/micropython_i2c/devices/muxes/__init__.py
"""I2C multiplexers: TCA9548A. Import each module directly.

Generated by ``dev_notebooks/package_inits_devNB``: edit the notebook, not this file.
"""

## `sim/__init__.py`
Host-only simulation. Nothing in it runs on the board: it uses CPython's `math` and `random`.

In [ ]:
%%writefile ../src/micropython_i2c/sim/__init__.py
"""Host-only simulation: run drivers against fake parts, in simulated time.

* ``sim.clock``: ``SimClock``, simulated time for every ``compat`` timing call.
* ``sim.signals``: quantities that change over time (ramps, steps, noise, lags).
* ``sim.bus``: ``FakeBus`` (a ``machine.I2C`` stand-in with mux routing) and generic parts.
* ``sim.bench``: a whole fake bench built from the same routing config as the real one.

Each part's fake sits next to its driver: ``devices/<type>/.../<part>_fake.py``.

Generated by ``dev_notebooks/package_inits_devNB``: edit the notebook, not this file.
"""

### validation/test

#### validation: nominal usage
Every `__init__` written above imports, and the package exposes its core names.

In [ ]:
PACKAGES = [
    "micropython_i2c",
    "micropython_i2c.devices.sensors",
    "micropython_i2c.devices.sensors.environmental",
    "micropython_i2c.devices.sensors.magnetic",
    "micropython_i2c.devices.sensors.distance",
    "micropython_i2c.devices.drivers",
    "micropython_i2c.devices.muxes",
    "micropython_i2c.sim",
]
for name in PACKAGES:
    mod = importlib.reload(importlib.import_module(name))
    print("{:48} {}".format(name, (mod.__doc__ or "").splitlines()[0]))

#### validation: anti-nominal usage
None: these files have no failure path of their own.

#### unit

In [ ]:
%%extract ../tests/test_package_inits.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize(
    "name",
    [
        "micropython_i2c",
        "micropython_i2c.devices.sensors",
        "micropython_i2c.devices.sensors.environmental",
        "micropython_i2c.devices.sensors.magnetic",
        "micropython_i2c.devices.sensors.distance",
        "micropython_i2c.devices.drivers",
        "micropython_i2c.devices.muxes",
        "micropython_i2c.sim",
    ],
)
def test_package_imports_with_a_docstring(name):
    mod = importlib.import_module(name)
    assert mod.__doc__ and "Generated by" in mod.__doc__

In [ ]:
%%extract ../tests/test_package_inits.py -a --strip-ipytest
%%ipytest


def test_core_names_are_exported():
    pkg = importlib.import_module("micropython_i2c")
    for name in pkg.__all__:
        assert hasattr(pkg, name), name
    assert "I2CConnection" in pkg.__all__